In [2]:
import yaml

from api_call import fetch_data_pacific, fetch_structure_pacific

import pycountry

In [3]:
population = fetch_data_pacific(
    source="DF_POP_DENSITY", 
    start_period="2024", 
    end_period="2024", 
    key="A..")

AttributeError: module 'sdmx' has no attribute 'Client'

In [4]:
population.head()

NameError: name 'population' is not defined

In [17]:
# print unique values in GEO_PICT column and INDICATOR column, as lists
print("Unique values in GEO_PICT column:", population['GEO_PICT'].unique().tolist())
print("Unique values in INDICATOR column:", population['INDICATOR'].unique().tolist())

Unique values in GEO_PICT column: ['_T', 'TO', 'PN', 'FM', 'AS', 'NR', 'GU', 'WS', 'TK', 'WF', 'POL', 'MP', '_TXPNG', 'NC', 'SB', 'CK', 'FJ', 'VU', 'PW', 'TV', 'KI', 'MH', 'MEL', 'PG', 'PF', 'MIC', 'NU', 'MELXPNG']
Unique values in INDICATOR column: ['LANDAREA', 'POPDENSITY', 'POPULATION']


In [18]:
population_ds = fetch_structure_pacific(source="DF_POP_DENSITY")

In [19]:
# save population_ds['dimensions']['GEO_PICT'] to a yaml file
with open('countries.yaml', 'w') as f:

    yaml.dump(population_ds['dimensions']['GEO_PICT'], f)


In [7]:
# create GEO column in population dataframe by mapping GEO_PICT values to their corresponding names in the population_ds dataframe
population['GEO'] = population['GEO_PICT'].map(population_ds['dimensions']['GEO_PICT'])

In [8]:
population.head()

,TIME_PERIOD,FREQ,GEO_PICT,INDICATOR,value,GEO
0,2024,A,_T,LANDAREA,540658.0,Pacific region
1,2024,A,TO,LANDAREA,720.0,Tonga
2,2024,A,PN,POPDENSITY,1.0,Pitcairn
3,2024,A,FM,POPULATION,112893.0,Micronesia (Federated States of)
4,2024,A,_T,POPDENSITY,26.0,Pacific region


In [9]:
print("Unique values in GEO column:", population['GEO'].unique().tolist())

Unique values in GEO column: ['Pacific region', 'Tonga', 'Pitcairn', 'Micronesia (Federated States of)', 'American Samoa', 'Nauru', 'Guam', 'Samoa', 'Tokelau', 'Wallis and Futuna', 'Polynesia', 'Northern Mariana Islands', 'Pacific region (excluding PNG)', 'New Caledonia', 'Solomon Islands', 'Cook Islands', 'Fiji', 'Vanuatu', 'Palau', 'Tuvalu', 'Kiribati', 'Marshall Islands', 'Melanesia', 'Papua New Guinea', 'French Polynesia', 'Micronesia', 'Niue', 'Melanesia (excluding PNG)']


In [10]:
to_ignore = ['MEL', 'MELXPNG', 'MIC', 'POL', '_T', '_TXPNG']

In [22]:
import plotly.graph_objects as go

# Filter to individual countries only
pacific_codes = [c for c in population['GEO_PICT'].unique() if c not in to_ignore]

# Map GEO_PICT (ISO alpha-2) to ISO alpha-3 for plotly
def to_iso3(alpha2):
    try:
        return pycountry.countries.get(alpha_2=alpha2).alpha_3
    except:
        return None

iso3_list = [to_iso3(c) for c in pacific_codes]
geo_names = [population_ds['dimensions']['GEO_PICT'].get(c, c) for c in pacific_codes]

# Drop codes pycountry couldn't resolve (e.g. Tokelau, Niue, Pitcairn)
pairs = [(i, g) for i, g in zip(iso3_list, geo_names) if i is not None]
iso3_list, geo_names = zip(*pairs) if pairs else ([], [])

fig = go.Figure(go.Choropleth(
    locations=list(iso3_list),
    z=[1] * len(iso3_list),
    text=list(geo_names),
    colorscale=[[0, '#4E91D2'], [1, "#FF0000"]],
    showscale=False,
    marker_line_color='white',
    marker_line_width=0.5,
    hovertemplate='%{text}<extra></extra>',
))

fig.update_geos(
    projection_type="natural earth",
    projection_rotation=dict(lon=180),
    showland=True,
    landcolor='#CCCCCC',
    showocean=True,
    oceancolor='#D6EAF8',
    showcoastlines=True,
    coastlinecolor='white',
    showframe=False,
    lataxis=dict(range=[-50, 35]),
)

fig.update_layout(
    title='Pacific Region',
    height=500,
    margin=dict(l=0, r=0, t=40, b=0),
)

fig.show()

In [25]:
import plotly.express as px

pop_df = (
    population[
        (population['INDICATOR'] == 'POPULATION') &
        (~population['GEO_PICT'].isin(to_ignore))
    ]
    .sort_values('value', ascending=False)
)

fig = px.bar(
    pop_df,
    x='GEO_PICT',
    y='value',
    title='Population by Pacific Country (2024)',
    labels={'GEO_PICT': 'Country', 'value': 'Population'},
)

fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [28]:
population[
        (population['INDICATOR'] == 'POPULATION') &
        (~population['GEO_PICT'].isin(to_ignore))
    ]

,TIME_PERIOD,FREQ,GEO_PICT,INDICATOR,value
3,2024,A,FM,POPULATION,112893.0
9,2024,A,TK,POPULATION,2453.0
10,2024,A,WF,POPULATION,11320.0
12,2024,A,MP,POPULATION,44668.0
17,2024,A,NC,POPULATION,291269.0
18,2024,A,SB,POPULATION,809539.0
19,2024,A,WS,POPULATION,217342.0
24,2024,A,FJ,POPULATION,926544.0
25,2024,A,VU,POPULATION,324085.0
28,2024,A,TO,POPULATION,104379.0
